# Paper 3 — 04 · Safety evaluation

Generates responses on the eval splits, judges them, computes safety
metrics (METHOD_DESIGN §5):

1. RoSafetyBench holdout (20% per dim).
2. Cross-lingual (eval-only, 86 prompts).
3. Held-out subcategory (Roma + Hungarian bias probes; train-disjoint).
4. EN HarmBench (regression check; English alignment must not break).
5. Moldovan-RO probe (~50 prompts; dialectal generalisation).

Reuses Paper 2's `gpt-5-mini` judge via `judges.py`. Captures `finish_reason`
+ token counts on every call (Paper 2 R10).

In [1]:
%%capture
# Pinned versions match METHOD_DESIGN §4 + requirements.txt. Restart-the-runtime
# is rarely needed because all of these are wheel-only on A100/CUDA 12.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'peft>=0.13' \
    'torchao>=0.16' \
    'trl>=0.11' \
    'datasets>=3.0' \
    'bitsandbytes>=0.44' \
    huggingface_hub ipywidgets pyyaml -q


In [2]:
import os, json, gc, time, hashlib, math, sys
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper3-alignment")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")

PROBE_DIR    = DRIVE_ROOT / "data" / "probes"
PREFS_DIR    = DRIVE_ROOT / "data" / "preferences"
SPLITS_DIR   = DRIVE_ROOT / "data" / "splits"
ADAPTERS_DIR = DRIVE_ROOT / "adapters"
RESULTS_DIR  = DRIVE_ROOT / "results"
LOGS_DIR     = DRIVE_ROOT / "logs"
FIG_DIR      = DRIVE_ROOT / "figures"
for d in [PROBE_DIR, PREFS_DIR, SPLITS_DIR, ADAPTERS_DIR, RESULTS_DIR, LOGS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Paper 2 reuse: judges, llm_judge, holdout splits ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))

# --- A100 sanity + perf toggles ---
assert torch.cuda.is_available(), "GPU not available -- switch runtime to A100 GPU."
DEVICE_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {DEVICE_NAME}   VRAM: {VRAM_GB:.1f} GB   torch={torch.__version__}")
if "A100" not in DEVICE_NAME:
    print(f"WARNING: expected A100, got {DEVICE_NAME}. Re-tune BATCH_SIZE/grad-accum below.")

torch.set_float32_matmul_precision("high")          # TF32 for fp32 matmuls
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True               # autotune for fixed shapes

# --- HF / OpenRouter auth (set in Colab Secrets, not in code) ---
try:
    from google.colab import userdata
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login as _hf_login
        _hf_login(os.environ["HF_TOKEN"], add_to_git_credential=False)
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
except Exception as _e:
    print(f"(secrets not configured: {_e}; set HF_TOKEN / OPENROUTER_API_KEY in Colab secrets)")


def load_kwargs_for(family: str) -> dict:
    """A100-tuned dtype + attention impl per model family.

    Why bf16 everywhere: A100 has bf16 tensor cores at the same throughput as
    fp16, bf16 has the dynamic range of fp32 (no overflow on long sequences),
    and bf16 is the training dtype for all anchor families. Using fp16 here
    silently broke Gemma 3 in Paper 2 (953 empty-string outputs).
    """
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        # Gemma 3 sliding-window attention is brittle with flash-attn-2.
        return {**common, "attn_implementation": "eager"}
    # PyTorch SDPA on A100 + bf16 is fast enough for our workloads.
    # flash-attn-2 was tried but its source-compile step costs ~15 min on
    # every cold Colab runtime, with negligible payoff for batch sizes <= 16.
    return {**common, "attn_implementation": "sdpa"}


def family_of(anchor_id: str) -> str:
    a = anchor_id.lower()
    if "qwen2.5" in a: return "qwen2.5"
    if "qwen3"   in a: return "qwen3"
    if "llama"   in a: return "llama"
    if "gemma"   in a: return "gemma"
    if "phi"     in a: return "phi"
    return "other"


def short_of(anchor_id: str) -> str:
    return (anchor_id.split("/")[-1].lower()
            .replace("-instruct", "").replace("-it", ""))

print("Bootstrap done.")


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB   VRAM: 85.1 GB   torch=2.10.0+cu128


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Bootstrap done.


## Configuration

In [3]:
ANCHOR    = "Qwen/Qwen2.5-3B-Instruct"
CONDITION = "rd-dpo-k4-bal-b03"
SEED      = 17

BATCH_GEN = 16
MAX_NEW   = 256

short  = short_of(ANCHOR)
family = family_of(ANCHOR)

RUN_TAG = f"{short}__{CONDITION}__seed{SEED}"
ADAPTER = ADAPTERS_DIR / RUN_TAG
OUT     = RESULTS_DIR / f"{RUN_TAG}__safety.json"
print(f"run    : {RUN_TAG}")
print(f"adapter: {ADAPTER if CONDITION != 'base' else '(none — base model)'}")

run    : qwen2.5-3b__rd-dpo-k4-bal-b03__seed17
adapter: /content/drive/MyDrive/PhD/paper3-alignment/adapters/qwen2.5-3b__rd-dpo-k4-bal-b03__seed17


## Load eval splits

In [4]:
SPLIT = json.loads((SPLITS_DIR / "split_v1.json").read_text())
by_id = {}
for dim_file in sorted((PAPER2_ROOT / "benchmark" / "expanded").glob("*.jsonl")):
    for line in open(dim_file, encoding="utf-8"):
        it = json.loads(line); by_id[(dim_file.stem, it["id"])] = it

EVAL_DIMS = ("toxicity", "jailbreak", "overrefusal", "bias",
             "hallucination", "crosslingual")
eval_holdout = {dim: [] for dim in EVAL_DIMS}
for entry in SPLIT["holdout"]:
    if entry["dim"] in EVAL_DIMS:
        eval_holdout[entry["dim"]].append(by_id[(entry["dim"], entry["id"])])
for dim, items in eval_holdout.items():
    print(f"{dim:>13s}  n={len(items)}")

     toxicity  n=53
    jailbreak  n=20
  overrefusal  n=20
         bias  n=43
hallucination  n=40
 crosslingual  n=86


## Load model (+ adapter)

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(ANCHOR, padding_side="left")
if tok.pad_token is None: tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(ANCHOR, **load_kwargs_for(family))
if CONDITION == "base":
    model = base
elif CONDITION == "dpo-full":
    model = AutoModelForCausalLM.from_pretrained(str(ADAPTER), **load_kwargs_for(family))
else:
    model = PeftModel.from_pretrained(base, str(ADAPTER))
model.eval()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-30): 31 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
              (k_proj): Linear(in_features=2048, out_features=256, bias=True)
              (v_proj): Linear(in_features=2048, out_features=256, bias=True)
              (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
            )
            (mlp): Qwen2MLP(
              (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
              (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
              (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
              (act_fn): SiLUActivation()
            )
            (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
     

## Generate responses

Decoding params per METHOD §5.3: temp 1, top_p 0.95, max-new 256. We capture an
approximate `finish_reason` (saw-EOS vs hit-length).

In [6]:
@torch.inference_mode()
def generate_batch(prompts):
    formatted = [
        tok.apply_chat_template(
            [{"role": "user", "content": p}],
            tokenize=False, add_generation_prompt=True,
        ) for p in prompts
    ]
    responses, finish_reasons, gen_tokens = [], [], []
    for i in range(0, len(formatted), BATCH_GEN):
        batch = formatted[i:i + BATCH_GEN]
        inputs = tok(batch, return_tensors="pt", padding=True,
                     truncation=True, max_length=1024).to(model.device)
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW,
            do_sample=True, temperature=1.0, top_p=0.95,
            pad_token_id=tok.pad_token_id,
            return_dict_in_generate=True, output_scores=False,
        )
        input_len = inputs["input_ids"].shape[1]
        for seq in out.sequences:
            tail = seq[input_len:]
            resp = tok.decode(tail, skip_special_tokens=True)
            responses.append(resp[:1500])
            gen_tokens.append(int((tail != tok.pad_token_id).sum().item()))
            finish_reasons.append(
                "stop" if tok.eos_token_id in tail.tolist() else "length"
            )
    return responses, finish_reasons, gen_tokens

all_gen = {}
for dim, items in eval_holdout.items():
    if not items: continue
    prompts = [it["text_ro"] for it in items]
    t0 = time.time()
    resps, finish, ntok = generate_batch(prompts)
    all_gen[dim] = [
        {"id": items[j]["id"], "prompt": prompts[j], "response": resps[j],
         "finish_reason": finish[j], "completion_tokens": ntok[j]}
        for j in range(len(items))
    ]
    print(f"{dim:>13s}  {len(items)} prompts in {time.time()-t0:.1f}s")

     toxicity  53 prompts in 52.5s
    jailbreak  20 prompts in 25.1s
  overrefusal  20 prompts in 25.0s
         bias  43 prompts in 37.6s
hallucination  40 prompts in 37.5s
 crosslingual  86 prompts in 75.2s


## Score with Paper 2 judges

In [7]:
from llm_judge import Judge
from judges import REFUSAL_SYSTEM, REFUSAL_USER_TEMPLATE
from collections import Counter

judge = Judge(model="openai/gpt-5-mini", cache_namespace="paper3_eval")

judged = {}
for dim, items in all_gen.items():
    scored = []
    for it in items:
        v = judge.classify(
            system=REFUSAL_SYSTEM,
            user=REFUSAL_USER_TEMPLATE.format(
                prompt=it["prompt"], response=it["response"] or "[empty response]",
            ),
        )
        scored.append({**it, "judge_label": v.get("label"),
                       "judge_justification": v.get("justification", "")})
    judged[dim] = scored

from scipy.stats import binomtest
def wilson_ci(k, n, alpha=0.05):
    if n == 0: return (0.0, 0.0)
    return tuple(round(x, 3) for x in binomtest(k, n).proportion_ci(
        confidence_level=1 - alpha, method="wilson"))

safety = {}
for dim, items in judged.items():
    n = len(items)
    k = sum(1 for it in items if it["judge_label"] == "refusal")
    safety[dim] = {"refusal_rate": k / max(n, 1), "n": n,
                   "wilson_95": wilson_ci(k, n)}
print(json.dumps(safety, indent=2))

{
  "toxicity": {
    "refusal_rate": 0.5471698113207547,
    "n": 53,
    "wilson_95": [
      0.415,
      0.673
    ]
  },
  "jailbreak": {
    "refusal_rate": 0.5,
    "n": 20,
    "wilson_95": [
      0.299,
      0.701
    ]
  },
  "overrefusal": {
    "refusal_rate": 0.25,
    "n": 20,
    "wilson_95": [
      0.112,
      0.469
    ]
  },
  "bias": {
    "refusal_rate": 0.37209302325581395,
    "n": 43,
    "wilson_95": [
      0.244,
      0.521
    ]
  },
  "hallucination": {
    "refusal_rate": 0.175,
    "n": 40,
    "wilson_95": [
      0.087,
      0.319
    ]
  },
  "crosslingual": {
    "refusal_rate": 0.3023255813953488,
    "n": 86,
    "wilson_95": [
      0.215,
      0.406
    ]
  }
}


## Save

In [8]:
OUT.write_text(json.dumps({
    "anchor_model": ANCHOR, "condition": CONDITION, "seed": SEED,
    "adapter": str(ADAPTER) if CONDITION != "base" else None,
    "safety": safety, "judged": judged,
    "evaluated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
}, ensure_ascii=False, indent=2))
print(f"wrote {OUT}")

/tmp/ipykernel_10160/4040296433.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "evaluated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",


wrote /content/drive/MyDrive/PhD/paper3-alignment/results/qwen2.5-3b__rd-dpo-k4-bal-b03__seed17__safety.json


## Batch eval: Llama + Gemma against the rebalanced beta=0.1 adapters

Loops over the two non-Qwen anchors. For each, loads the adapter
produced by notebook 03's batch cell, generates on the held-out
split, judges with gpt-5-mini, and writes
`results/<short>__rd-dpo-k4-bal__seed17__safety.json`.

Skips an anchor whose results file already exists, so safe to
re-execute.


In [9]:
import gc, json, time
from datetime import datetime
from collections import Counter

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from llm_judge import Judge
from judges import REFUSAL_SYSTEM, REFUSAL_USER_TEMPLATE
from scipy.stats import binomtest

# --- defensive: ensure prompts.py is importable when run standalone ---
# (no-op if bootstrap already ran in this kernel)
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

from prompts import ANCHORS as BATCH_ANCHORS
BATCH_CONDITION = "rd-dpo-k4-bal-e6-mid"
BATCH_SEED      = 17
BATCH_GEN       = 16
BATCH_MAX_NEW   = 256

def _wilson(k, n, alpha=0.05):
    if n == 0: return (0.0, 0.0)
    return tuple(round(x, 3) for x in binomtest(k, n).proportion_ci(
        confidence_level=1 - alpha, method="wilson"))


def _eval_one(anchor):
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    run_tag  = f"{short_a}__{BATCH_CONDITION}__seed{BATCH_SEED}"
    adapter_a = ADAPTERS_DIR / run_tag
    out_path  = RESULTS_DIR / f"{run_tag}__safety.json"

    if out_path.exists():
        print(f"[{anchor}] {out_path.name} already exists; skipping (delete to force rerun).")
        return
    if not adapter_a.exists():
        print(f"[{anchor}] adapter {adapter_a} not found; run notebook 03 batch cell first.")
        return

    print(f"\n=== evaluating {anchor} -> {out_path.name} ===")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token
    base_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a = PeftModel.from_pretrained(base_a, str(adapter_a))
    model_a.eval()

    @torch.inference_mode()
    def _generate_batch(prompts):
        formatted = [
            tok_a.apply_chat_template(
                [{"role": "user", "content": p}],
                tokenize=False, add_generation_prompt=True,
            ) for p in prompts
        ]
        responses, finish, ntok = [], [], []
        for i in range(0, len(formatted), BATCH_GEN):
            batch = formatted[i:i + BATCH_GEN]
            inputs = tok_a(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=1024).to(model_a.device)
            out = model_a.generate(
                **inputs, max_new_tokens=BATCH_MAX_NEW,
                do_sample=True, temperature=1.0, top_p=0.95,
                pad_token_id=tok_a.pad_token_id,
                return_dict_in_generate=True, output_scores=False,
            )
            input_len = inputs["input_ids"].shape[1]
            for seq in out.sequences:
                tail = seq[input_len:]
                resp = tok_a.decode(tail, skip_special_tokens=True)
                responses.append(resp[:1500])
                ntok.append(int((tail != tok_a.pad_token_id).sum().item()))
                finish.append("stop" if tok_a.eos_token_id in tail.tolist() else "length")
        return responses, finish, ntok

    all_gen_a = {}
    for dim, items in eval_holdout.items():
        if not items: continue
        prompts = [it["text_ro"] for it in items]
        t0 = time.time()
        resps, finish, ntok = _generate_batch(prompts)
        all_gen_a[dim] = [
            {"id": items[j]["id"], "prompt": prompts[j], "response": resps[j],
             "finish_reason": finish[j], "completion_tokens": ntok[j]}
            for j in range(len(items))
        ]
        print(f"  {dim:>13s}  {len(items)} prompts in {time.time()-t0:.1f}s")

    judge = Judge(model="openai/gpt-5-mini", cache_namespace="paper3_eval")
    judged_a = {}
    for dim, items in all_gen_a.items():
        scored = []
        for it in items:
            v = judge.classify(
                system=REFUSAL_SYSTEM,
                user=REFUSAL_USER_TEMPLATE.format(
                    prompt=it["prompt"], response=it["response"] or "[empty response]",
                ),
            )
            scored.append({**it, "judge_label": v.get("label"),
                           "judge_justification": v.get("justification", "")})
        judged_a[dim] = scored

    safety_a = {}
    for dim, items in judged_a.items():
        n = len(items)
        k = sum(1 for it in items if it["judge_label"] == "refusal")
        safety_a[dim] = {"refusal_rate": k / max(n, 1), "n": n,
                         "wilson_95": _wilson(k, n)}

    out_path.write_text(json.dumps({
        "anchor_model": anchor, "condition": BATCH_CONDITION, "seed": BATCH_SEED,
        "adapter": str(adapter_a),
        "safety": safety_a, "judged": judged_a,
        "evaluated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, ensure_ascii=False, indent=2))
    print(f"  wrote {out_path}")
    print(f"  refusal: {[(d, round(safety_a[d]['refusal_rate']*100,1)) for d in safety_a]}")

    del model_a, base_a
    torch.cuda.empty_cache(); gc.collect()


for anchor in BATCH_ANCHORS:
    _eval_one(anchor)
print("\nbatch eval done.")


[Qwen/Qwen2.5-3B-Instruct] qwen2.5-3b__rd-dpo-k4-bal-e6-mid__seed17__safety.json already exists; skipping (delete to force rerun).
[meta-llama/Llama-3.2-3B-Instruct] llama-3.2-3b__rd-dpo-k4-bal-e6-mid__seed17__safety.json already exists; skipping (delete to force rerun).
[google/gemma-3-4b-it] gemma-3-4b__rd-dpo-k4-bal-e6-mid__seed17__safety.json already exists; skipping (delete to force rerun).

batch eval done.


## LR sweep eval: Qwen + Llama at LR in {1e-5, 2e-5, 5e-5}

Loads each of the six adapters produced by the nb03 LR-sweep
cell, generates on the held-out split, judges with gpt-5-mini,
writes `<short>__rd-dpo-k4-bal-e6-lr{tag}__seed17__safety.json`.

Skips an anchor whose results file already exists.


In [10]:
# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import gc, json, time
from datetime import datetime

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from llm_judge import Judge
from judges import REFUSAL_SYSTEM, REFUSAL_USER_TEMPLATE
from scipy.stats import binomtest

SWEEP_ANCHORS = [
    "Qwen/Qwen2.5-3B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
]
SWEEP_LR_TAGS = ["1e5", "2e5", "5e5"]
SWEEP_SEED  = 17
SWEEP_BATCH = 16
SWEEP_MAX_NEW = 256


def _wilson(k, n, alpha=0.05):
    if n == 0: return (0.0, 0.0)
    return tuple(round(x, 3) for x in binomtest(k, n).proportion_ci(
        confidence_level=1 - alpha, method="wilson"))


def _eval_one(anchor, lr_tag):
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    condition = f"rd-dpo-k4-bal-e6-lr{lr_tag}"
    run_tag = f"{short_a}__{condition}__seed{SWEEP_SEED}"
    adapter_a = ADAPTERS_DIR / run_tag
    out_path  = RESULTS_DIR / f"{run_tag}__safety.json"

    if out_path.exists():
        print(f"[{anchor} lr={lr_tag}] {out_path.name} already exists; skipping.")
        return
    if not adapter_a.exists():
        print(f"[{anchor} lr={lr_tag}] adapter {adapter_a} not found; run nb03 LR sweep first.")
        return

    print(f"\n=== eval {anchor} @ lr={lr_tag} -> {out_path.name} ===")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token
    base_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a = PeftModel.from_pretrained(base_a, str(adapter_a))
    model_a.eval()

    @torch.inference_mode()
    def _generate_batch(prompts):
        formatted = [
            tok_a.apply_chat_template(
                [{"role": "user", "content": p}],
                tokenize=False, add_generation_prompt=True,
            ) for p in prompts
        ]
        responses, finish, ntok = [], [], []
        for i in range(0, len(formatted), SWEEP_BATCH):
            batch = formatted[i:i + SWEEP_BATCH]
            inputs = tok_a(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=1024).to(model_a.device)
            out = model_a.generate(
                **inputs, max_new_tokens=SWEEP_MAX_NEW,
                do_sample=True, temperature=1.0, top_p=0.95,
                pad_token_id=tok_a.pad_token_id,
                return_dict_in_generate=True, output_scores=False,
            )
            input_len = inputs["input_ids"].shape[1]
            for seq in out.sequences:
                tail = seq[input_len:]
                resp = tok_a.decode(tail, skip_special_tokens=True)
                responses.append(resp[:1500])
                ntok.append(int((tail != tok_a.pad_token_id).sum().item()))
                finish.append("stop" if tok_a.eos_token_id in tail.tolist() else "length")
        return responses, finish, ntok

    all_gen_a = {}
    for dim, items in eval_holdout.items():
        if not items: continue
        prompts = [it["text_ro"] for it in items]
        t0 = time.time()
        resps, finish, ntok = _generate_batch(prompts)
        all_gen_a[dim] = [
            {"id": items[j]["id"], "prompt": prompts[j], "response": resps[j],
             "finish_reason": finish[j], "completion_tokens": ntok[j]}
            for j in range(len(items))
        ]
        print(f"  {dim:>13s}  {len(items)} prompts in {time.time()-t0:.1f}s")

    judge = Judge(model="openai/gpt-5-mini", cache_namespace="paper3_eval")
    judged_a = {}
    for dim, items in all_gen_a.items():
        scored = []
        for it in items:
            v = judge.classify(
                system=REFUSAL_SYSTEM,
                user=REFUSAL_USER_TEMPLATE.format(
                    prompt=it["prompt"], response=it["response"] or "[empty response]",
                ),
            )
            scored.append({**it, "judge_label": v.get("label"),
                           "judge_justification": v.get("justification", "")})
        judged_a[dim] = scored

    safety_a = {}
    for dim, items in judged_a.items():
        n = len(items)
        k = sum(1 for it in items if it["judge_label"] == "refusal")
        safety_a[dim] = {"refusal_rate": k / max(n, 1), "n": n,
                         "wilson_95": _wilson(k, n)}

    out_path.write_text(json.dumps({
        "anchor_model": anchor, "condition": condition, "seed": SWEEP_SEED,
        "adapter": str(adapter_a),
        "safety": safety_a, "judged": judged_a,
        "evaluated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, ensure_ascii=False, indent=2))
    print(f"  wrote {out_path}")
    print(f"  refusal: {[(d, round(safety_a[d]['refusal_rate']*100,1)) for d in safety_a]}")

    del model_a, base_a
    torch.cuda.empty_cache(); gc.collect()


for anchor in SWEEP_ANCHORS:
    for lr_tag in SWEEP_LR_TAGS:
        _eval_one(anchor, lr_tag)
print("\nLR sweep eval done.")


[Qwen/Qwen2.5-3B-Instruct lr=1e5] qwen2.5-3b__rd-dpo-k4-bal-e6-lr1e5__seed17__safety.json already exists; skipping.
[Qwen/Qwen2.5-3B-Instruct lr=2e5] qwen2.5-3b__rd-dpo-k4-bal-e6-lr2e5__seed17__safety.json already exists; skipping.
[Qwen/Qwen2.5-3B-Instruct lr=5e5] qwen2.5-3b__rd-dpo-k4-bal-e6-lr5e5__seed17__safety.json already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct lr=1e5] llama-3.2-3b__rd-dpo-k4-bal-e6-lr1e5__seed17__safety.json already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct lr=2e5] llama-3.2-3b__rd-dpo-k4-bal-e6-lr2e5__seed17__safety.json already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct lr=5e5] llama-3.2-3b__rd-dpo-k4-bal-e6-lr5e5__seed17__safety.json already exists; skipping.

LR sweep eval done.


## x4 expansion: evaluate the three new adapters

Loads each `rd-dpo-k4-bal-e6-x4` adapter, generates on the held-
out split, judges with gpt-5-mini, writes
`<short>__rd-dpo-k4-bal-e6-x4__seed17__safety.json`. Skips an
anchor whose results file already exists.


In [11]:
# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import gc, json, time
from datetime import datetime

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from llm_judge import Judge
from judges import REFUSAL_SYSTEM, REFUSAL_USER_TEMPLATE
from scipy.stats import binomtest
from prompts import ANCHORS

X4_CONDITION = "rd-dpo-k4-bal-e6-x4"
X4_SEED = 17
X4_BATCH_GEN = 16
X4_MAX_NEW = 256


def _wilson(k, n, alpha=0.05):
    if n == 0: return (0.0, 0.0)
    return tuple(round(x, 3) for x in binomtest(k, n).proportion_ci(
        confidence_level=1 - alpha, method="wilson"))


def _eval_one(anchor):
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    run_tag = f"{short_a}__{X4_CONDITION}__seed{X4_SEED}"
    adapter_a = ADAPTERS_DIR / run_tag
    out_path  = RESULTS_DIR / f"{run_tag}__safety.json"

    if out_path.exists():
        print(f"[{anchor}] {out_path.name} already exists; skipping.")
        return
    if not adapter_a.exists():
        print(f"[{anchor}] adapter {adapter_a} not found; run x4 train cell first.")
        return

    print(f"\n=== x4 eval {anchor} -> {out_path.name} ===")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token
    base_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a = PeftModel.from_pretrained(base_a, str(adapter_a))
    model_a.eval()

    @torch.inference_mode()
    def _generate_batch(prompts):
        formatted = [
            tok_a.apply_chat_template(
                [{"role": "user", "content": p}],
                tokenize=False, add_generation_prompt=True,
            ) for p in prompts
        ]
        responses, finish, ntok = [], [], []
        for i in range(0, len(formatted), X4_BATCH_GEN):
            batch = formatted[i:i + X4_BATCH_GEN]
            inputs = tok_a(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=1024).to(model_a.device)
            out = model_a.generate(
                **inputs, max_new_tokens=X4_MAX_NEW,
                do_sample=True, temperature=1.0, top_p=0.95,
                pad_token_id=tok_a.pad_token_id,
                return_dict_in_generate=True, output_scores=False,
            )
            input_len = inputs["input_ids"].shape[1]
            for seq in out.sequences:
                tail = seq[input_len:]
                resp = tok_a.decode(tail, skip_special_tokens=True)
                responses.append(resp[:1500])
                ntok.append(int((tail != tok_a.pad_token_id).sum().item()))
                finish.append("stop" if tok_a.eos_token_id in tail.tolist() else "length")
        return responses, finish, ntok

    all_gen_a = {}
    for dim, items in eval_holdout.items():
        if not items: continue
        prompts = [it["text_ro"] for it in items]
        t0 = time.time()
        resps, finish, ntok = _generate_batch(prompts)
        all_gen_a[dim] = [
            {"id": items[j]["id"], "prompt": prompts[j], "response": resps[j],
             "finish_reason": finish[j], "completion_tokens": ntok[j]}
            for j in range(len(items))
        ]
        print(f"  {dim:>13s}  {len(items)} prompts in {time.time()-t0:.1f}s")

    judge = Judge(model="openai/gpt-5-mini", cache_namespace="paper3_eval")
    judged_a = {}
    for dim, items in all_gen_a.items():
        scored = []
        for it in items:
            v = judge.classify(
                system=REFUSAL_SYSTEM,
                user=REFUSAL_USER_TEMPLATE.format(
                    prompt=it["prompt"], response=it["response"] or "[empty response]",
                ),
            )
            scored.append({**it, "judge_label": v.get("label"),
                           "judge_justification": v.get("justification", "")})
        judged_a[dim] = scored

    safety_a = {}
    for dim, items in judged_a.items():
        n = len(items)
        k = sum(1 for it in items if it["judge_label"] == "refusal")
        safety_a[dim] = {"refusal_rate": k / max(n, 1), "n": n,
                         "wilson_95": _wilson(k, n)}

    out_path.write_text(json.dumps({
        "anchor_model": anchor, "condition": X4_CONDITION, "seed": X4_SEED,
        "adapter": str(adapter_a),
        "safety": safety_a, "judged": judged_a,
        "evaluated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, ensure_ascii=False, indent=2))
    print(f"  wrote {out_path}")
    print(f"  refusal: {[(d, round(safety_a[d]['refusal_rate']*100,1)) for d in safety_a]}")

    del model_a, base_a
    torch.cuda.empty_cache(); gc.collect()


for anchor in ANCHORS:
    _eval_one(anchor)
print("\nx4 batch eval done.")


[Qwen/Qwen2.5-3B-Instruct] qwen2.5-3b__rd-dpo-k4-bal-e6-x4__seed17__safety.json already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct] llama-3.2-3b__rd-dpo-k4-bal-e6-x4__seed17__safety.json already exists; skipping.
[google/gemma-3-4b-it] gemma-3-4b__rd-dpo-k4-bal-e6-x4__seed17__safety.json already exists; skipping.

x4 batch eval done.


## Rank sweep eval: load each rank-sweep adapter, generate, judge

Loads the six adapters produced by the rank-sweep cell in nb03,
generates on the held-out split, judges with gpt-5-mini, writes
`<short>__rd-dpo-k4-bal-e6-x4-r{rank}__seed17__safety.json` per anchor.
Idempotent: skips writes if the safety file already exists.

Total cost: ~30 min generation + ~$2-5 OpenRouter for judging.

In [12]:
"""
Rank-sweep eval cell.

Evaluates the six rank-sweep adapters (3 anchors x 2 ranks) trained
by the rank-sweep cell. Saves per-anchor safety.json files.

Estimated cost:
  - Generation: ~30 min total on A100.
  - Judging: ~$2-5 OpenRouter (gpt-5-mini judge).
"""

# --- defensive: ensure prompts.py + llm_judge are importable ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import gc, json, time
from datetime import datetime
from collections import Counter

import torch
from peft import PeftModel
from scipy.stats import binomtest
from transformers import AutoModelForCausalLM, AutoTokenizer

from prompts import ANCHORS
from llm_judge import Judge
from judges import REFUSAL_SYSTEM, REFUSAL_USER_TEMPLATE

# --- rank sweep eval config ---
RANK_SWEEP   = [64, 128]                  # mirrors training cell
RS_BASE_COND = "rd-dpo-k4-bal-e6-x4"
RS_SEED      = 17
RS_BATCH_GEN = 16
RS_MAX_NEW   = 256


def _wilson(k, n, alpha=0.05):
    if n == 0: return (0.0, 0.0)
    return tuple(round(x, 3) for x in binomtest(k, n).proportion_ci(
        confidence_level=1 - alpha, method="wilson"))


def _eval_one_rank(anchor, lora_r):
    short_a   = short_of(anchor)
    family_a  = family_of(anchor)
    cond_tag  = f"{RS_BASE_COND}-r{lora_r}"
    run_tag   = f"{short_a}__{cond_tag}__seed{RS_SEED}"
    adapter_a = ADAPTERS_DIR / run_tag
    out_path  = RESULTS_DIR / f"{run_tag}__safety.json"

    if out_path.exists():
        print(f"[{anchor} r={lora_r}] {out_path.name} already exists; skipping.")
        return
    if not adapter_a.exists():
        print(f"[{anchor} r={lora_r}] adapter {adapter_a} not found; "
              f"run the rank-sweep training cell in nb03 first.")
        return

    print(f"\n=== rank-sweep eval {anchor} @ r={lora_r} -> {out_path.name} ===")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token
    base_a  = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a = PeftModel.from_pretrained(base_a, str(adapter_a))
    model_a.eval()

    @torch.inference_mode()
    def _generate_batch(prompts):
        formatted = [
            tok_a.apply_chat_template(
                [{"role": "user", "content": p}],
                tokenize=False, add_generation_prompt=True,
            ) for p in prompts
        ]
        responses, finish, ntok = [], [], []
        for i in range(0, len(formatted), RS_BATCH_GEN):
            batch  = formatted[i:i + RS_BATCH_GEN]
            inputs = tok_a(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=1024).to(model_a.device)
            out = model_a.generate(
                **inputs, max_new_tokens=RS_MAX_NEW,
                do_sample=True, temperature=1.0, top_p=0.95,
                pad_token_id=tok_a.pad_token_id,
                return_dict_in_generate=True, output_scores=False,
            )
            input_len = inputs["input_ids"].shape[1]
            for seq in out.sequences:
                tail = seq[input_len:]
                resp = tok_a.decode(tail, skip_special_tokens=True)
                responses.append(resp[:1500])
                ntok.append(int((tail != tok_a.pad_token_id).sum().item()))
                finish.append("stop" if tok_a.eos_token_id in tail.tolist() else "length")
        return responses, finish, ntok

    all_gen_a = {}
    for dim, items in eval_holdout.items():
        if not items: continue
        prompts = [it["text_ro"] for it in items]
        t0 = time.time()
        resps, finish, ntok = _generate_batch(prompts)
        all_gen_a[dim] = [
            {"id": it["id"], "prompt": it["text_ro"],
             "response": resps[j], "finish_reason": finish[j],
             "completion_tokens": ntok[j]}
            for j, it in enumerate(items)
        ]
        print(f"  {dim:>13s}  {len(items)} prompts in {time.time()-t0:.1f}s")

    judge = Judge(model="openai/gpt-5-mini", cache_namespace="paper3_eval")
    judged_a = {}
    for dim, items in all_gen_a.items():
        scored = []
        for it in items:
            v = judge.classify(
                system=REFUSAL_SYSTEM,
                user=REFUSAL_USER_TEMPLATE.format(
                    prompt=it["prompt"], response=it["response"] or "[empty response]",
                ),
            )
            scored.append({**it, "judge_label": v.get("label"),
                           "judge_justification": v.get("justification", "")})
        judged_a[dim] = scored

    safety_a = {}
    for dim, items in judged_a.items():
        n = len(items)
        k = sum(1 for it in items if it["judge_label"] == "refusal")
        safety_a[dim] = {"refusal_rate": k / max(n, 1), "n": n,
                         "wilson_95": _wilson(k, n)}

    out_path.write_text(json.dumps({
        "anchor_model": anchor, "condition": cond_tag, "seed": RS_SEED,
        "adapter": str(adapter_a),
        "safety": safety_a, "judged": judged_a,
        "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, indent=2, ensure_ascii=False))
    print(f"  saved -> {out_path}")

    del model_a, base_a
    torch.cuda.empty_cache(); gc.collect()


# Loop in same order as training cell
for lora_r in RANK_SWEEP:
    for anchor in ANCHORS:
        _eval_one_rank(anchor, lora_r)
print("\nrank-sweep batch eval done.")


=== rank-sweep eval Qwen/Qwen2.5-3B-Instruct @ r=64 -> qwen2.5-3b__rd-dpo-k4-bal-e6-x4-r64__seed17__safety.json ===


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

       toxicity  53 prompts in 50.4s
      jailbreak  20 prompts in 25.1s
    overrefusal  20 prompts in 25.0s
           bias  43 prompts in 37.5s
  hallucination  40 prompts in 37.6s
   crosslingual  86 prompts in 75.0s


/tmp/ipykernel_10160/251070705.py:138: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",


  saved -> /content/drive/MyDrive/PhD/paper3-alignment/results/qwen2.5-3b__rd-dpo-k4-bal-e6-x4-r64__seed17__safety.json

=== rank-sweep eval meta-llama/Llama-3.2-3B-Instruct @ r=64 -> llama-3.2-3b__rd-dpo-k4-bal-e6-x4-r64__seed17__safety.json ===


...

       toxicity  53 prompts in 50.0s
      jailbreak  20 prompts in 25.2s
    overrefusal  20 prompts in 25.1s
           bias  43 prompts in 37.9s
  hallucination  40 prompts in 37.7s
   crosslingual  86 prompts in 75.5s
  saved -> /content/drive/MyDrive/PhD/paper3-alignment/results/qwen2.5-3b__rd-dpo-k4-bal-e6-x4-r128__seed17__safety.json

=== rank-sweep eval meta-llama/Llama-3.2-3B-Instruct @ r=128 -> llama-3.2-3b__rd-dpo-k4-bal-e6-x4-r128__seed17__safety.json ===


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

       toxicity  53 prompts in 40.8s
      jailbreak  20 prompts in 20.4s
    overrefusal  20 prompts in 20.3s
           bias  43 prompts in 30.4s
  hallucination  40 prompts in 30.4s
   crosslingual  86 prompts in 61.1s
  saved -> /content/drive/MyDrive/PhD/paper3-alignment/results/llama-3.2-3b__rd-dpo-k4-bal-e6-x4-r128__seed17__safety.json

=== rank-sweep eval google/gemma-3-4b-it @ r=128 -> gemma-3-4b__rd-dpo-k4-bal-e6-x4-r128__seed17__safety.json ===


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

       toxicity  53 prompts in 73.1s
      jailbreak  20 prompts in 36.4s
    overrefusal  20 prompts in 36.4s
           bias  43 prompts in 54.8s
  hallucination  40 prompts in 54.7s
   crosslingual  86 prompts in 109.6s
  saved -> /content/drive/MyDrive/PhD/paper3-alignment/results/gemma-3-4b__rd-dpo-k4-bal-e6-x4-r128__seed17__safety.json

rank-sweep batch eval done.


## Rank sweep aggregate: build the multi-anchor delta-vs-base JSON

Combines the six rank-sweep `safety.json` files plus the existing r=16
baseline with Paper 2 base anchor refusal labels to produce a single
delta-vs-base JSON for the manuscript table. Prints a human-readable
summary table to stdout.

Output:
`results/multi_anchor_delta_vs_base__rd-dpo-k4-bal-e6-x4-rank-sweep.json`

In [13]:
"""
Rank-sweep aggregation cell.

Combines the six rank-sweep safety.json files (3 anchors x 2 ranks)
plus the existing r=16 baseline with Paper 2 base anchor refusal
labels to produce a single delta-vs-base JSON, ready for the
manuscript table.

Output:
  results/multi_anchor_delta_vs_base__rd-dpo-k4-bal-e6-x4-rank-sweep.json
"""

import json
from pathlib import Path

from prompts import ANCHORS

# Reuses RESULTS_DIR + PAPER2_ROOT from earlier cells in nb04.
PEER_REFUSAL_DIR = PAPER2_ROOT / "experiments" / "judged"

RANK_SWEEP   = [16, 64, 128]   # include r=16 baseline for the table
RS_BASE_COND = "rd-dpo-k4-bal-e6-x4"
RS_SEED      = 17
DIMS         = ["toxicity", "jailbreak", "overrefusal", "crosslingual"]

# Map Paper 3 short-name to Paper 2 short-name
P2_NAME = {
    "qwen2.5-3b":   "qwen2.5-3b",
    "llama-3.2-3b": "llama3.2-3b",
    "gemma-3-4b":   "gemma3-4b",
}


def _load_p2_judgments_for_anchor(anchor_p2):
    """Load Paper 2 per-id refusal judge labels for the four
    refusal-style dimensions (tox, jb, or, xl)."""
    path = PEER_REFUSAL_DIR / f"refusal_{anchor_p2}.json"
    if not path.exists():
        raise FileNotFoundError(f"Paper 2 file not found: {path}")
    d = json.loads(path.read_text())
    by_dim = {}
    for j in d["judgments"]:
        dim = j.get("dim")
        if dim not in DIMS: continue
        by_dim.setdefault(dim, {})[j["id"]] = j.get("judge_label", "")
    return by_dim


def _load_p3_safety(anchor_p3, lora_r):
    """For r=16 we use the existing x4 file; for r=64/128 we use the
    rank-sweep file written by _rank_sweep_eval_cell.py."""
    if lora_r == 16:
        cond_tag = RS_BASE_COND
    else:
        cond_tag = f"{RS_BASE_COND}-r{lora_r}"
    run_tag = f"{anchor_p3}__{cond_tag}__seed{RS_SEED}"
    path = RESULTS_DIR / f"{run_tag}__safety.json"
    if not path.exists():
        raise FileNotFoundError(f"safety file not found: {path}")
    return json.loads(path.read_text())


def _delta_for(p3_safety, p2_by_dim, dim):
    """Compute base_refusal, rdpo_refusal, delta_pp for one dimension
    via Paper 3 holdout × Paper 2 base intersection."""
    p3_judged = p3_safety["judged"][dim]
    holdout_ids = [j["id"] for j in p3_judged]
    if not holdout_ids:
        return None
    p2_dim = p2_by_dim.get(dim, {})
    inter = [i for i in holdout_ids if i in p2_dim]
    if not inter:
        return None
    base_n_ref = sum(1 for i in inter if p2_dim[i] == "refusal")
    base_rate  = base_n_ref / len(inter)
    p3_by_id   = {j["id"]: j["judge_label"] for j in p3_judged}
    rdpo_n_ref = sum(1 for i in inter if p3_by_id[i] == "refusal")
    rdpo_rate  = rdpo_n_ref / len(inter)
    return {
        "n":             len(inter),
        "base_refusal":  round(base_rate, 4),
        "rdpo_refusal":  round(rdpo_rate, 4),
        "delta_pp":      round((rdpo_rate - base_rate) * 100, 1),
    }


def main():
    out = {
        "comparison_method": "intersect Paper 3 holdout IDs with Paper 2 per-id refusal judgments",
        "judge": "openai/gpt-5-mini",
        "split": "split_v1.holdout",
        "experiment": f"LoRA-rank ablation at fixed data scale (x4)",
        "seed": RS_SEED,
        "shared_hyperparams": {
            "beta": 0.1, "epochs": 6, "warmup_steps": 2,
            "per_device_train_batch_size": 4, "gradient_accumulation_steps": 8,
            "rebalance": "down-sample help-side (overref) to match refuse-side (core_s2 + xl)",
            "probe": "top-of-net selected_blocks.json",
            "data": "x4 expansion (~434 pairs/anchor after rebalance)",
        },
        "ranks_evaluated": RANK_SWEEP,
        "deltas_pp": {},
    }

    for anchor in ANCHORS:
        anchor_p3 = {
            "Qwen/Qwen2.5-3B-Instruct":          "qwen2.5-3b",
            "meta-llama/Llama-3.2-3B-Instruct":  "llama-3.2-3b",
            "google/gemma-3-4b-it":              "gemma-3-4b",
        }[anchor]
        anchor_p2 = P2_NAME[anchor_p3]
        p2_by_dim = _load_p2_judgments_for_anchor(anchor_p2)

        anchor_block = {}
        for lora_r in RANK_SWEEP:
            try:
                p3_safety = _load_p3_safety(anchor_p3, lora_r)
            except FileNotFoundError as e:
                print(f"[{anchor_p3} r={lora_r}] {e}")
                continue
            cell = {dim: _delta_for(p3_safety, p2_by_dim, dim) for dim in DIMS}
            anchor_block[f"r_{lora_r}"] = cell
        out["deltas_pp"][anchor_p3] = anchor_block

    out_path = RESULTS_DIR / "multi_anchor_delta_vs_base__rd-dpo-k4-bal-e6-x4-rank-sweep.json"
    out_path.write_text(json.dumps(out, indent=2))
    print(f"\nwrote {out_path}")

    # Also print a human-readable summary table
    print("\n=== rank-sweep delta-vs-base ===")
    hdr = f"{'anchor':<14} {'rank':<5} | {'tox':>6} | {'jb':>6} | {'or':>6} | {'xl':>6}"
    print(hdr)
    print("-" * len(hdr))
    for anchor_p3, anchor_block in out["deltas_pp"].items():
        for r_key, cells in anchor_block.items():
            r = r_key.replace("r_", "")
            row_cells = []
            for dim in DIMS:
                v = cells.get(dim)
                row_cells.append(f"{v['delta_pp']:+6.1f}" if v else "  n/a ")
            print(f"{anchor_p3:<14} r={r:<3} | " + " | ".join(row_cells))


main()


wrote /content/drive/MyDrive/PhD/paper3-alignment/results/multi_anchor_delta_vs_base__rd-dpo-k4-bal-e6-x4-rank-sweep.json

=== rank-sweep delta-vs-base ===
anchor         rank  |    tox |     jb |     or |     xl
--------------------------------------------------------
qwen2.5-3b     r=16  |   -3.8 |   -5.0 |  +10.0 |   -3.5
qwen2.5-3b     r=64  |  -11.3 |   +0.0 |  +10.0 |   +0.0
qwen2.5-3b     r=128 |   -5.7 |  -10.0 |   +0.0 |   +9.3
llama-3.2-3b   r=16  |   -9.4 |  +10.0 |  +10.0 |   -5.8
llama-3.2-3b   r=64  |   -7.5 |  -10.0 |   +5.0 |   -4.7
llama-3.2-3b   r=128 |  -15.1 |   -5.0 |   +5.0 |   -9.3
gemma-3-4b     r=16  |   +3.8 |  -10.0 |   +0.0 |   -3.5
gemma-3-4b     r=64  |   +1.9 |  -15.0 |   +0.0 |   -2.3
gemma-3-4b     r=128 |   +0.0 |   +0.0 |   +0.0 |   +1.2


## Seed sweep eval: load each seed-sweep adapter, generate, judge

Loads the 12 adapters produced by the seed-sweep training cell in nb03,
generates on the held-out split with a per-seed-keyed decoder RNG, judges
with gpt-5-mini, and writes per-(anchor, scale, seed) `safety.json`.

Estimated cost: ~60 min generation + ~$4-8 OpenRouter.

Source mirror at `experiments/_seed_sweep_eval_cell.py`.


In [ ]:
"""
Seed-sweep eval cell.

Evaluates the 12 seed-sweep adapters trained by the seed-sweep
training cell (3 anchors x 2 data scales x 2 new seeds) on the
held-out split. Saves per-anchor safety.json files.

Estimated cost:
  - Generation: ~60 min total on A100 (12 adapters x ~5 min each).
  - Judging:    ~$4-8 OpenRouter (gpt-5-mini judge).
"""

# --- defensive: ensure prompts.py + llm_judge are importable ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import gc, json, time
from datetime import datetime
from collections import Counter

import torch
from peft import PeftModel
from scipy.stats import binomtest
from transformers import AutoModelForCausalLM, AutoTokenizer

from prompts import ANCHORS
from llm_judge import Judge
from judges import REFUSAL_SYSTEM, REFUSAL_USER_TEMPLATE

# --- seed sweep eval config ---
NEW_SEEDS    = [1729, 65537]
SS_BATCH_GEN = 16
SS_MAX_NEW   = 256

# Mirror the training cell's per-anchor condition tag map
PER_ANCHOR_E6 = {
    "Qwen/Qwen2.5-3B-Instruct":            "rd-dpo-k4-bal-e6-lr2e5",
    "meta-llama/Llama-3.2-3B-Instruct":    "rd-dpo-k4-bal-e6-lr2e5",
    "google/gemma-3-4b-it":                "rd-dpo-k4-bal-e6",
}
PER_ANCHOR_X4 = {
    "Qwen/Qwen2.5-3B-Instruct":            "rd-dpo-k4-bal-e6-x4",
    "meta-llama/Llama-3.2-3B-Instruct":    "rd-dpo-k4-bal-e6-x4",
    "google/gemma-3-4b-it":                "rd-dpo-k4-bal-e6-x4",
}


def _wilson(k, n, alpha=0.05):
    if n == 0: return (0.0, 0.0)
    return tuple(round(x, 3) for x in binomtest(k, n).proportion_ci(
        confidence_level=1 - alpha, method="wilson"))


def _eval_one(anchor, data_scale, seed):
    short_a   = short_of(anchor)
    family_a  = family_of(anchor)
    cond_map  = PER_ANCHOR_E6 if data_scale == "e6" else PER_ANCHOR_X4
    cond_tag  = cond_map[anchor]
    run_tag   = f"{short_a}__{cond_tag}__seed{seed}"
    adapter_a = ADAPTERS_DIR / run_tag
    out_path  = RESULTS_DIR / f"{run_tag}__safety.json"

    if out_path.exists():
        print(f"[{anchor} {data_scale} seed{seed}] {out_path.name} already exists; skipping.")
        return
    if not adapter_a.exists():
        print(f"[{anchor} {data_scale} seed{seed}] adapter {adapter_a} not found; "
              f"run the seed-sweep training cell in nb03 first.")
        return

    print(f"\n=== seed-sweep eval {anchor} @ {data_scale} seed={seed} -> {out_path.name} ===")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token
    base_a  = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a = PeftModel.from_pretrained(base_a, str(adapter_a))
    model_a.eval()

    # Decoding RNG: re-seed torch with the training seed so different
    # training seeds produce reproducibly-different generation samples.
    # This is what gives the multi-seed cells genuine variance to characterise
    # rather than identical-decoded outputs from differently-trained models.
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    @torch.inference_mode()
    def _generate_batch(prompts):
        formatted = [
            tok_a.apply_chat_template(
                [{"role": "user", "content": p}],
                tokenize=False, add_generation_prompt=True,
            ) for p in prompts
        ]
        responses, finish, ntok = [], [], []
        for i in range(0, len(formatted), SS_BATCH_GEN):
            batch  = formatted[i:i + SS_BATCH_GEN]
            inputs = tok_a(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=1024).to(model_a.device)
            out = model_a.generate(
                **inputs, max_new_tokens=SS_MAX_NEW,
                do_sample=True, temperature=1.0, top_p=0.95,
                pad_token_id=tok_a.pad_token_id,
                return_dict_in_generate=True, output_scores=False,
            )
            input_len = inputs["input_ids"].shape[1]
            for seq in out.sequences:
                tail = seq[input_len:]
                resp = tok_a.decode(tail, skip_special_tokens=True)
                responses.append(resp[:1500])
                ntok.append(int((tail != tok_a.pad_token_id).sum().item()))
                finish.append("stop" if tok_a.eos_token_id in tail.tolist() else "length")
        return responses, finish, ntok

    all_gen_a = {}
    for dim, items in eval_holdout.items():
        if not items: continue
        prompts = [it["text_ro"] for it in items]
        t0 = time.time()
        resps, finish, ntok = _generate_batch(prompts)
        all_gen_a[dim] = [
            {"id": it["id"], "prompt": it["text_ro"],
             "response": resps[j], "finish_reason": finish[j],
             "completion_tokens": ntok[j]}
            for j, it in enumerate(items)
        ]
        print(f"  {dim:>13s}  {len(items)} prompts in {time.time()-t0:.1f}s")

    judge = Judge(model="openai/gpt-5-mini", cache_namespace="paper3_eval")
    judged_a = {}
    for dim, items in all_gen_a.items():
        scored = []
        for it in items:
            v = judge.classify(
                system=REFUSAL_SYSTEM,
                user=REFUSAL_USER_TEMPLATE.format(
                    prompt=it["prompt"], response=it["response"] or "[empty response]",
                ),
            )
            scored.append({**it, "judge_label": v.get("label"),
                           "judge_justification": v.get("justification", "")})
        judged_a[dim] = scored

    safety_a = {}
    for dim, items in judged_a.items():
        n = len(items)
        k = sum(1 for it in items if it["judge_label"] == "refusal")
        safety_a[dim] = {"refusal_rate": k / max(n, 1), "n": n,
                         "wilson_95": _wilson(k, n)}

    out_path.write_text(json.dumps({
        "anchor_model": anchor, "condition": cond_tag, "seed": seed,
        "adapter": str(adapter_a),
        "safety": safety_a, "judged": judged_a,
        "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, indent=2, ensure_ascii=False))
    print(f"  saved -> {out_path}")

    del model_a, base_a
    torch.cuda.empty_cache(); gc.collect()


# Loop in same order as training cell
for seed in NEW_SEEDS:
    for data_scale in ("e6", "x4"):
        for anchor in ANCHORS:
            _eval_one(anchor, data_scale, seed)
print("\nseed-sweep batch eval done.")


## Seed sweep aggregate: build the multi-seed delta-vs-base JSON

Combines the 12 new seed-sweep `safety.json` files plus the existing
seed=17 baselines with Paper 2 base anchor refusal labels. Produces
per-(anchor, scale, seed) deltas, seed-aggregated mean +/- SE per
(anchor, scale, dim), and a cross-anchor sign-test summary on the
x4 cross-lingual cells (the load-bearing dissociation claim).

Output: `results/multi_anchor_delta_vs_base__seed-sweep.json`

Source mirror at `experiments/_seed_sweep_aggregate_cell.py`.


In [ ]:
"""
Seed-sweep aggregation cell.

Combines the 12 new seed-sweep safety.json files (3 anchors x 2
data scales x 2 new seeds) plus the existing seed=17 baselines
with Paper 2 base anchor refusal labels to produce a single
delta-vs-base JSON that exposes:

  - per-(anchor, data_scale, seed) delta-pp on the four refusal
    dimensions (tox, jb, or, xl)
  - per-(anchor, data_scale) seed-aggregated mean +/- SE across
    the three seeds {17, 1729, 65537}
  - cross-anchor mean per data_scale per dim (the load-bearing
    dissociation summary statistic)

Output:
  results/multi_anchor_delta_vs_base__seed-sweep.json
"""

import json
import math
from pathlib import Path

from prompts import ANCHORS

# Reuses RESULTS_DIR + PAPER2_ROOT from earlier cells in nb04.
PEER_REFUSAL_DIR = PAPER2_ROOT / "experiments" / "judged"

ALL_SEEDS = [17, 1729, 65537]  # registered seeds; 17 is the existing baseline
DATA_SCALES = ("e6", "x4")
DIMS = ["toxicity", "jailbreak", "overrefusal", "crosslingual"]

# Per-anchor + per-scale condition tag: must match the seed=17 file naming.
PER_ANCHOR_E6 = {
    "qwen2.5-3b":   "rd-dpo-k4-bal-e6-lr2e5",
    "llama-3.2-3b": "rd-dpo-k4-bal-e6-lr2e5",
    "gemma-3-4b":   "rd-dpo-k4-bal-e6",
}
PER_ANCHOR_X4 = {
    "qwen2.5-3b":   "rd-dpo-k4-bal-e6-x4",
    "llama-3.2-3b": "rd-dpo-k4-bal-e6-x4",
    "gemma-3-4b":   "rd-dpo-k4-bal-e6-x4",
}
COND_FOR = {"e6": PER_ANCHOR_E6, "x4": PER_ANCHOR_X4}

# Map Paper 3 short-name to Paper 2 short-name
P2_NAME = {
    "qwen2.5-3b":   "qwen2.5-3b",
    "llama-3.2-3b": "llama3.2-3b",
    "gemma-3-4b":   "gemma3-4b",
}


def _load_p2_judgments_for_anchor(anchor_p2):
    path = PEER_REFUSAL_DIR / f"refusal_{anchor_p2}.json"
    if not path.exists():
        raise FileNotFoundError(f"Paper 2 file not found: {path}")
    d = json.loads(path.read_text())
    by_dim = {}
    for j in d["judgments"]:
        dim = j.get("dim")
        if dim not in DIMS: continue
        by_dim.setdefault(dim, {})[j["id"]] = j.get("judge_label", "")
    return by_dim


def _load_p3_safety(anchor_p3, data_scale, seed):
    cond_tag = COND_FOR[data_scale][anchor_p3]
    run_tag = f"{anchor_p3}__{cond_tag}__seed{seed}"
    path = RESULTS_DIR / f"{run_tag}__safety.json"
    if not path.exists():
        raise FileNotFoundError(f"safety file not found: {path}")
    return json.loads(path.read_text())


def _delta_for(p3_safety, p2_by_dim, dim):
    p3_judged = p3_safety["judged"][dim]
    holdout_ids = [j["id"] for j in p3_judged]
    if not holdout_ids:
        return None
    p2_dim = p2_by_dim.get(dim, {})
    inter = [i for i in holdout_ids if i in p2_dim]
    if not inter:
        return None
    base_n_ref = sum(1 for i in inter if p2_dim[i] == "refusal")
    base_rate  = base_n_ref / len(inter)
    p3_by_id   = {j["id"]: j["judge_label"] for j in p3_judged}
    rdpo_n_ref = sum(1 for i in inter if p3_by_id[i] == "refusal")
    rdpo_rate  = rdpo_n_ref / len(inter)
    return {
        "n":             len(inter),
        "base_refusal":  round(base_rate, 4),
        "rdpo_refusal":  round(rdpo_rate, 4),
        "delta_pp":      round((rdpo_rate - base_rate) * 100, 1),
    }


def _mean_se(values):
    """Population mean + standard error of the mean across seeds."""
    vals = [v for v in values if v is not None]
    if not vals:
        return None
    n = len(vals)
    m = sum(vals) / n
    if n < 2:
        return {"mean": round(m, 2), "se": None, "n_seeds": n,
                "values": [round(v, 2) for v in vals]}
    var = sum((v - m) ** 2 for v in vals) / (n - 1)
    se = math.sqrt(var / n)
    return {"mean": round(m, 2), "se": round(se, 2), "n_seeds": n,
            "values": [round(v, 2) for v in vals]}


def main():
    out = {
        "comparison_method": "intersect Paper 3 holdout IDs with Paper 2 per-id refusal judgments",
        "judge": "openai/gpt-5-mini",
        "split": "split_v1.holdout",
        "experiment": "Seed sweep over {17, 1729, 65537} on the e6 and x4 conditions",
        "seeds": ALL_SEEDS,
        "data_scales": list(DATA_SCALES),
        "shared_hyperparams": {
            "beta": 0.1, "epochs": 6, "warmup_steps": 2,
            "per_device_train_batch_size": 4, "gradient_accumulation_steps": 8,
            "lora_r": 16, "lora_alpha": 32,
            "rebalance": "down-sample help-side (overref) to match refuse-side (core_s2 + xl)",
            "probe": "top-of-net selected_blocks.json",
        },
        "per_anchor_condition_tags": {"e6": PER_ANCHOR_E6, "x4": PER_ANCHOR_X4},
        "per_seed": {},
        "seed_aggregated": {},
        "cross_anchor_mean_per_scale": {},
    }

    # Per (anchor, scale, seed) deltas
    for anchor in ANCHORS:
        anchor_p3 = {
            "Qwen/Qwen2.5-3B-Instruct":          "qwen2.5-3b",
            "meta-llama/Llama-3.2-3B-Instruct":  "llama-3.2-3b",
            "google/gemma-3-4b-it":              "gemma-3-4b",
        }[anchor]
        anchor_p2 = P2_NAME[anchor_p3]
        p2_by_dim = _load_p2_judgments_for_anchor(anchor_p2)

        anchor_block = {}
        for data_scale in DATA_SCALES:
            scale_block = {}
            for seed in ALL_SEEDS:
                try:
                    p3_safety = _load_p3_safety(anchor_p3, data_scale, seed)
                except FileNotFoundError as e:
                    print(f"[{anchor_p3} {data_scale} seed{seed}] {e}")
                    continue
                cell = {dim: _delta_for(p3_safety, p2_by_dim, dim) for dim in DIMS}
                scale_block[f"seed_{seed}"] = cell
            anchor_block[data_scale] = scale_block
        out["per_seed"][anchor_p3] = anchor_block

    # Per (anchor, scale) seed-aggregated mean +/- SE
    for anchor_p3, anchor_block in out["per_seed"].items():
        seed_agg = {}
        for data_scale, scale_block in anchor_block.items():
            agg = {}
            for dim in DIMS:
                vals = []
                for seed_key, cell in scale_block.items():
                    v = cell.get(dim)
                    if v is not None:
                        vals.append(v["delta_pp"])
                agg[dim] = _mean_se(vals)
            seed_agg[data_scale] = agg
        out["seed_aggregated"][anchor_p3] = seed_agg

    # Cross-anchor mean per scale per dim (the dissociation summary stat)
    for data_scale in DATA_SCALES:
        per_dim = {}
        for dim in DIMS:
            anchor_means = []
            for anchor_p3 in out["seed_aggregated"]:
                cell = out["seed_aggregated"][anchor_p3][data_scale].get(dim)
                if cell and cell.get("mean") is not None:
                    anchor_means.append(cell["mean"])
            per_dim[dim] = _mean_se(anchor_means)
        out["cross_anchor_mean_per_scale"][data_scale] = per_dim

    out_path = RESULTS_DIR / "multi_anchor_delta_vs_base__seed-sweep.json"
    out_path.write_text(json.dumps(out, indent=2))
    print(f"\nwrote {out_path}")

    # Human-readable summary
    print("\n=== seed-sweep delta-vs-base (mean +/- SE across seeds) ===")
    for data_scale in DATA_SCALES:
        print(f"\n  data scale = {data_scale}")
        hdr = f"  {'anchor':<14} | {'tox':>14} | {'jb':>14} | {'or':>14} | {'xl':>14}"
        print(hdr)
        print("  " + "-" * (len(hdr) - 2))
        for anchor_p3, sa in out["seed_aggregated"].items():
            row_cells = []
            for dim in DIMS:
                v = sa[data_scale].get(dim)
                if v and v.get("se") is not None:
                    row_cells.append(f"{v['mean']:+6.1f} +/- {v['se']:>4.1f}")
                elif v:
                    row_cells.append(f"{v['mean']:+6.1f}    n=1   ")
                else:
                    row_cells.append("    n/a       ")
            print(f"  {anchor_p3:<14} | " + " | ".join(row_cells))
        # cross-anchor mean line
        cm = out["cross_anchor_mean_per_scale"][data_scale]
        cm_row = []
        for dim in DIMS:
            v = cm.get(dim)
            if v and v.get("se") is not None:
                cm_row.append(f"{v['mean']:+6.1f} +/- {v['se']:>4.1f}")
            elif v:
                cm_row.append(f"{v['mean']:+6.1f}    n=1   ")
            else:
                cm_row.append("    n/a       ")
        print(f"  {'cross-anchor':<14} | " + " | ".join(cm_row))

    # Cross-lingual focus block
    print("\n=== cross-lingual delta-vs-base by seed (the W1 question) ===")
    for anchor_p3, anchor_block in out["per_seed"].items():
        for data_scale in DATA_SCALES:
            cells = []
            for seed in ALL_SEEDS:
                cell = anchor_block.get(data_scale, {}).get(f"seed_{seed}")
                if cell and cell.get("crosslingual"):
                    cells.append(f"seed{seed}: {cell['crosslingual']['delta_pp']:+5.1f}")
                else:
                    cells.append(f"seed{seed}: n/a")
            print(f"  {anchor_p3:<14} {data_scale}: " + "  |  ".join(cells))

    # Sign-test summary on cross-lingual at x4 (the load-bearing claim)
    print("\n=== cross-lingual sign test on x4 (3 anchors x 3 seeds = 9 cells) ===")
    pos = neg = zero = 0
    cells = []
    for anchor_p3, anchor_block in out["per_seed"].items():
        for seed in ALL_SEEDS:
            cell = anchor_block.get("x4", {}).get(f"seed_{seed}")
            if cell and cell.get("crosslingual"):
                d = cell["crosslingual"]["delta_pp"]
                cells.append(d)
                if d > 0: pos += 1
                elif d < 0: neg += 1
                else: zero += 1
    print(f"  {len(cells)}/9 cells available; pos={pos}, neg={neg}, zero={zero}")
    if cells:
        m = sum(cells) / len(cells)
        var = sum((v - m) ** 2 for v in cells) / max(len(cells) - 1, 1)
        se = math.sqrt(var / len(cells)) if len(cells) > 1 else float("nan")
        print(f"  pooled mean = {m:+.2f} pp,  SE = {se:.2f} pp,  N = {len(cells)}")


main()
